# 11 · Urban-to-peripheral thermal gradient — Figure 7

**Study:** Retrospective LST modeling and derived SUHI diagnostics in Barranquilla  
**Scope:** Retrospective component, 2013–2025  
**Purpose:** Reconstruct the observed and predicted urban-to-peripheral thermal gradient for 2023–2025 and generate the final figure supporting the 4–6 km non-urban reference band.

### Frozen conventions
- Panel (a): `Urban core → 0–2 → 2–4 → 4–6 → 6–8 km`.
- Panel (b): peripheral bands only: `0–2 → 2–4 → 4–6 → 6–8 km`.
- The `8–10 km` band is retained for audit purposes but excluded from the principal figure because its support is strongly truncated.
- Color identifies TEST year; line/marker style distinguishes observed and predicted values.
- The 4–6 km band is the fixed non-urban reference \(R\).


## 00. Entorno y montaje


In [ ]:

# ============================================================
# 00. Google Drive — montaje robusto para Google Colab
# ============================================================

from pathlib import Path
import os
import shutil

from google.colab import drive

MOUNT_POINT = Path("/content/drive")
PROJECT_DIR = (
    MOUNT_POINT
    / "MyDrive"
    / "Barranquilla_LST_2013_2026_Modeling"
)

def is_active_mount(mount_point="/content/drive"):
    """Comprueba si el punto aparece como filesystem montado."""
    with open("/proc/mounts", "r", encoding="utf-8") as f:
        mounts = f.read().splitlines()
    return any(
        line.split()[1] == mount_point
        for line in mounts
        if len(line.split()) >= 2
    )

if not is_active_mount():

    # Colab puede dejar una carpeta residual /content/drive
    # después de reiniciar el runtime.
    if MOUNT_POINT.exists() and any(MOUNT_POINT.iterdir()):
        stale = Path("/content/drive_stale")

        if stale.exists():
            shutil.rmtree(stale)

        MOUNT_POINT.rename(stale)
        print(f"Carpeta residual movida a: {stale}")

    drive.mount(str(MOUNT_POINT))

print("Drive montado:", is_active_mount())
print("Proyecto disponible:", PROJECT_DIR.exists())

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"No se encontró el proyecto en:\n{PROJECT_DIR}"
    )


## 01. Reconstrucción del gradiente y auditoría


In [ ]:
# ============================================================
# G3 — PASO 1
# Tabla maestra para gradiente térmico urbano–periférico
#
# Salidas:
#   - urban/peripheral medians observed vs predicted
#   - thermal contrast relative to 4–6 km reference
#
# Aún NO genera figura.
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

from scipy import ndimage


# ============================================================
# 1. RUTAS
# ============================================================

BASE = Path(
    "/content/drive/MyDrive/"
    "Barranquilla_LST_2013_2026_Modeling"
)

LST_DIR = (
    BASE
    / "07_models"
    / "t3_climate"
    / "module08_independent_2016_2025"
    / "09_physical_unit_evaluation_and_exports"
    / "rasters_celsius"
)

MASK_DIR = (
    BASE
    / "04_quality_masks"
    / "common_valid_mask"
)

LAND_PATH = (
    BASE
    / "04_quality_masks"
    / "final_land_domain"
    / "land_domain_final_EXP10B_V5.tif"
)

FRACTION_PATH = (
    BASE
    / "01_aoi_reference"
    / "emc_built_2022"
    / "processed"
    / "EMC_BUILT_fraction_E2022_EPSG32618_30m.tif"
)

YEARS = [2023, 2024, 2025]


# ============================================================
# 2. FUNCIONES
# ============================================================

def read_raster(path):

    if not path.exists():
        raise FileNotFoundError(
            f"No existe:\n{path}"
        )

    with rasterio.open(path) as src:

        arr = src.read(1).astype("float64")

        nodata = src.nodata

        if nodata is not None:
            arr[arr == nodata] = np.nan

        meta = {
            "crs": src.crs,
            "shape": src.shape,
            "transform": src.transform,
            "res": src.res,
            "bounds": src.bounds,
        }

    return arr, meta


def read_mask(path):

    if not path.exists():
        raise FileNotFoundError(
            f"No existe:\n{path}"
        )

    with rasterio.open(path) as src:

        arr = src.read(1)

        meta = {
            "crs": src.crs,
            "shape": src.shape,
            "transform": src.transform,
        }

    return arr > 0, meta


# ============================================================
# 3. GRILLA DE REFERENCIA
# ============================================================

obs_2025, ref_meta = read_raster(
    LST_DIR
    / "M5_observed_LST_C_2025.tif"
)

assert ref_meta["crs"].to_epsg() == 32618
assert ref_meta["shape"] == (1115, 1257)


# ============================================================
# 4. SOPORTE TEST FIJO
# ============================================================

land_mask, land_meta = read_mask(
    LAND_PATH
)

assert land_meta["shape"] == ref_meta["shape"]
assert land_meta["transform"] == ref_meta["transform"]

year_masks = []

for year in YEARS:

    mask_path = (
        MASK_DIR
        / f"EXP10Bv6_common_valid_mask_{year}.tif"
    )

    m, m_meta = read_mask(
        mask_path
    )

    assert m_meta["shape"] == ref_meta["shape"]
    assert m_meta["transform"] == ref_meta["transform"]

    year_masks.append(m)


fixed_test_support = (
    land_mask
    & year_masks[0]
    & year_masks[1]
    & year_masks[2]
)

n_fixed = int(
    fixed_test_support.sum()
)

print(
    "Fixed TEST support:",
    n_fixed
)

assert n_fixed == 906563


# ============================================================
# 5. RECONSTRUIR BU_all Y BU_core
# ============================================================

fraction, fraction_meta = read_raster(
    FRACTION_PATH
)

assert fraction_meta["shape"] == ref_meta["shape"]
assert fraction_meta["transform"] == ref_meta["transform"]

BU_THRESHOLD = 0.10
CORE_MIN_AREA_KM2 = 0.333

PIXEL_AREA_KM2 = (
    30.0 * 30.0
    / 1e6
)

BU_all = (
    np.isfinite(fraction)
    & (fraction >= BU_THRESHOLD)
)

structure = np.ones(
    (3, 3),
    dtype=int
)

labels, _ = ndimage.label(
    BU_all,
    structure=structure
)

counts = np.bincount(
    labels.ravel()
)

areas_km2 = (
    counts
    * PIXEL_AREA_KM2
)

keep_labels = np.where(
    areas_km2
    >= CORE_MIN_AREA_KM2
)[0]

keep_labels = (
    keep_labels[
        keep_labels != 0
    ]
)

BU_core = np.isin(
    labels,
    keep_labels
)

print(
    "BU_core components:",
    len(keep_labels)
)

assert len(keep_labels) == 18


# ============================================================
# 6. DISTANCIA A BU_core
# ============================================================

distance_km = ndimage.distance_transform_edt(
    ~BU_core,
    sampling=0.03
)


# ============================================================
# 7. BANDAS PERIFÉRICAS
# ============================================================

peripheral_support = (
    fixed_test_support
    & (~BU_all)
)

band_defs = {
    "0–2 km": (0.0, 2.0),
    "2–4 km": (2.0, 4.0),
    "4–6 km": (4.0, 6.0),
    "6–8 km": (6.0, 8.0),
    "8–10 km": (8.0, 10.0),
}

band_masks = {}

for name, (lo, hi) in band_defs.items():

    mask = (
        peripheral_support
        & (distance_km > lo)
        & (distance_km <= hi)
    )

    band_masks[name] = mask


# ------------------------------------------------------------
# Auditoría geométrica
# ------------------------------------------------------------

expected_counts = {
    "0–2 km": 298410,
    "2–4 km": 239190,
    "4–6 km": 167527,
    "6–8 km": 67076,
    "8–10 km": 3199,
}

print("\nBand geometry")
print("-------------")

for name, mask in band_masks.items():

    n = int(
        mask.sum()
    )

    print(
        name,
        "->",
        n
    )

    assert (
        n
        == expected_counts[name]
    )


# ============================================================
# 8. SOPORTE URBANO U
# ============================================================

U_geometry = (
    BU_core
    & fixed_test_support
)


# ============================================================
# 9. TABLA MAESTRA
# ============================================================

rows = []

for year in YEARS:

    obs, obs_meta = read_raster(
        LST_DIR
        / f"M5_observed_LST_C_{year}.tif"
    )

    pred, pred_meta = read_raster(
        LST_DIR
        / f"M5_predicted_LST_C_{year}.tif"
    )

    assert obs_meta["shape"] == ref_meta["shape"]
    assert pred_meta["shape"] == ref_meta["shape"]
    assert obs_meta["transform"] == ref_meta["transform"]
    assert pred_meta["transform"] == ref_meta["transform"]

    finite_pair = (
        np.isfinite(obs)
        & np.isfinite(pred)
    )

    # --------------------------------------------------------
    # Urban support
    # --------------------------------------------------------

    u_mask = (
        U_geometry
        & finite_pair
    )

    rows.append({
        "year": year,
        "zone": "Urban core",
        "zone_order": 0,
        "n_pixels": int(u_mask.sum()),
        "median_obs_c": float(
            np.median(obs[u_mask])
        ),
        "median_pred_c": float(
            np.median(pred[u_mask])
        ),
    })

    # --------------------------------------------------------
    # Peripheral bands
    # --------------------------------------------------------

    for order, name in enumerate(
        [
            "0–2 km",
            "2–4 km",
            "4–6 km",
            "6–8 km",
            "8–10 km",
        ],
        start=1
    ):

        mask = (
            band_masks[name]
            & finite_pair
        )

        rows.append({
            "year": year,
            "zone": name,
            "zone_order": order,
            "n_pixels": int(mask.sum()),
            "median_obs_c": float(
                np.median(obs[mask])
            ),
            "median_pred_c": float(
                np.median(pred[mask])
            ),
        })


gradient = pd.DataFrame(
    rows
)


# ============================================================
# 10. TREF Y CONTRASTE RESPECTO A 4–6 KM
# ============================================================

gradient["contrast_obs_c"] = np.nan
gradient["contrast_pred_c"] = np.nan

for year in YEARS:

    tref_obs = float(
        gradient.loc[
            (gradient["year"] == year)
            & (gradient["zone"] == "4–6 km"),
            "median_obs_c"
        ].iloc[0]
    )

    tref_pred = float(
        gradient.loc[
            (gradient["year"] == year)
            & (gradient["zone"] == "4–6 km"),
            "median_pred_c"
        ].iloc[0]
    )

    sel = (
        gradient["year"]
        == year
    )

    gradient.loc[
        sel,
        "contrast_obs_c"
    ] = (
        gradient.loc[
            sel,
            "median_obs_c"
        ]
        - tref_obs
    )

    gradient.loc[
        sel,
        "contrast_pred_c"
    ] = (
        gradient.loc[
            sel,
            "median_pred_c"
        ]
        - tref_pred
    )


# ============================================================
# 11. VALIDAR MEDIANAS OBSERVADAS YA CONGELADAS
# ============================================================

expected_obs = {
    2023: {
        "0–2 km": 38.242,
        "2–4 km": 36.490,
        "4–6 km": 36.446,
        "6–8 km": 36.817,
        "8–10 km": 30.847,
    },

    2024: {
        "0–2 km": 36.702,
        "2–4 km": 35.348,
        "4–6 km": 35.499,
        "6–8 km": 35.873,
        "8–10 km": 31.025,
    },

    2025: {
        "0–2 km": 34.344,
        "2–4 km": 32.893,
        "4–6 km": 32.848,
        "6–8 km": 33.328,
        "8–10 km": 29.808,
    },
}

for year, values in expected_obs.items():

    for zone, expected in values.items():

        obtained = float(
            gradient.loc[
                (gradient["year"] == year)
                & (gradient["zone"] == zone),
                "median_obs_c"
            ].iloc[0]
        )

        assert np.isclose(
            obtained,
            expected,
            atol=0.002
        ), (
            f"Mismatch {year} {zone}: "
            f"{obtained:.4f} vs {expected:.4f}"
        )


print(
    "\nAUDITORÍA DEL GRADIENTE OBSERVADO: OK"
)


# ============================================================
# 12. MOSTRAR TABLA
# ============================================================

gradient = gradient.sort_values(
    ["year", "zone_order"]
).reset_index(drop=True)

display(
    gradient.round(4)
)

# ============================================================
# 13. EXPORTAR TABLA FINAL DEL GRADIENTE
# ============================================================

OUTPUT_DIR = BASE / "03_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRADIENT_CSV = OUTPUT_DIR / "thermal_gradient.csv"

gradient.to_csv(
    GRADIENT_CSV,
    index=False
)

print("Tabla guardada:", GRADIENT_CSV)


## 02. Figura editorial final


In [ ]:
# ============================================================
# G3 — FIGURA EDITORIAL FINAL
# Urban-to-peripheral thermal gradient
#
# Requiere:
#   dataframe "gradient" generado previamente
#
# Diseño editorial:
#   - coherente con Figure 4 del manuscrito
#   - color = TEST year
#   - solid + circle = Observed
#   - dashed + square = Predicted
#   - leyenda común superior
#
# Panel (a):
#   Urban core + 0–2 + 2–4 + 4–6 + 6–8 km
#
# Panel (b):
#   SOLO bandas periféricas:
#   0–2 + 2–4 + 4–6 + 6–8 km
#
# 8–10 km:
#   excluida deliberadamente por soporte espacial truncado
# ============================================================

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

BASE = Path(
    "/content/drive/MyDrive/"
    "Barranquilla_LST_2013_2026_Modeling"
)

FIG_DIR = (
    BASE
    / "03_outputs"
    / "figures"
)

FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EXPORT_DPI = 600


# ============================================================
# 2. ESTILO EDITORIAL
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 120,
    "savefig.dpi": EXPORT_DPI,
})


# ------------------------------------------------------------
# Color = año
# Mantiene continuidad visual con Figure 4
# ------------------------------------------------------------

year_colors = {
    2023: "tab:blue",
    2024: "tab:orange",
    2025: "tab:green",
}


# ------------------------------------------------------------
# Estilo = tipo de producto
# ------------------------------------------------------------

OBS_LINEWIDTH = 2.0
PRED_LINEWIDTH = 1.8

OBS_MARKERSIZE = 5.5
PRED_MARKERSIZE = 5.0


# ============================================================
# 3. PANEL (a): ZONAS
# ============================================================

zones_a = [
    "Urban core",
    "0–2 km",
    "2–4 km",
    "4–6 km",
    "6–8 km",
]

zone_labels_a = [
    "Urban\ncore",
    "0–2\nkm",
    "2–4\nkm",
    "4–6\nkm",
    "6–8\nkm",
]

zone_order_a = {
    zone: i
    for i, zone in enumerate(zones_a)
}

x_a = np.arange(
    len(zones_a)
)


# ============================================================
# 4. PANEL (b): SOLO BANDAS PERIFÉRICAS
# ============================================================

zones_b = [
    "0–2 km",
    "2–4 km",
    "4–6 km",
    "6–8 km",
]

zone_labels_b = [
    "0–2\nkm",
    "2–4\nkm",
    "4–6\nkm",
    "6–8\nkm",
]

zone_order_b = {
    zone: i
    for i, zone in enumerate(zones_b)
}

x_b = np.arange(
    len(zones_b)
)


# ============================================================
# 5. PREPARAR DATOS PANEL (a)
# ============================================================

plot_df_a = (
    gradient[
        gradient["zone"].isin(zones_a)
    ]
    .copy()
)

plot_df_a["x"] = (
    plot_df_a["zone"]
    .map(zone_order_a)
)

plot_df_a = (
    plot_df_a
    .sort_values(
        ["year", "x"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 6. PREPARAR DATOS PANEL (b)
# ============================================================

plot_df_b = (
    gradient[
        gradient["zone"].isin(zones_b)
    ]
    .copy()
)

plot_df_b["x"] = (
    plot_df_b["zone"]
    .map(zone_order_b)
)

plot_df_b = (
    plot_df_b
    .sort_values(
        ["year", "x"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 7. AUDITORÍA
# ============================================================

assert len(plot_df_a) == 15
assert len(plot_df_b) == 12

for year in [2023, 2024, 2025]:

    tmp_a = (
        plot_df_a[
            plot_df_a["year"] == year
        ]
    )

    tmp_b = (
        plot_df_b[
            plot_df_b["year"] == year
        ]
    )

    assert list(
        tmp_a["zone"]
    ) == zones_a

    assert list(
        tmp_b["zone"]
    ) == zones_b

    for col in [
        "median_obs_c",
        "median_pred_c",
        "contrast_obs_c",
        "contrast_pred_c",
    ]:

        assert np.all(
            np.isfinite(
                tmp_a[col]
            )
        )

        assert np.all(
            np.isfinite(
                tmp_b[col]
            )
        )


print(
    "AUDITORÍA DATOS PARA FIGURA v3: OK"
)


# ============================================================
# 8. CREAR FIGURA
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13.2, 5.4)
)

# Más espacio superior para la leyenda común
fig.subplots_adjust(
    left=0.07,
    right=0.985,
    bottom=0.17,
    top=0.84,
    wspace=0.24
)


# ============================================================
# 9. FRANJA DE REFERENCIA R
# ============================================================

# Más tenue que en la versión anterior
R_FACE = "0.88"
R_ALPHA = 0.52


# ------------------------------------------------------------
# Panel (a): 4–6 km = x = 3
# ------------------------------------------------------------

R_X_A = 3

axes[0].axvspan(
    R_X_A - 0.34,
    R_X_A + 0.34,
    facecolor=R_FACE,
    edgecolor="none",
    alpha=R_ALPHA,
    zorder=0
)


# ------------------------------------------------------------
# Panel (b): 4–6 km = x = 2
# ------------------------------------------------------------

R_X_B = 2

axes[1].axvspan(
    R_X_B - 0.34,
    R_X_B + 0.34,
    facecolor=R_FACE,
    edgecolor="none",
    alpha=R_ALPHA,
    zorder=0
)


# ============================================================
# 10. PANEL (a)
# Median LST along urban-to-peripheral gradient
# ============================================================

ax = axes[0]

for year in [2023, 2024, 2025]:

    tmp = (
        plot_df_a[
            plot_df_a["year"] == year
        ]
        .sort_values("x")
    )

    color = year_colors[year]

    # --------------------------------------------------------
    # Observed
    # --------------------------------------------------------

    ax.plot(
        tmp["x"],
        tmp["median_obs_c"],
        color=color,
        linestyle="-",
        linewidth=OBS_LINEWIDTH,
        marker="o",
        markersize=OBS_MARKERSIZE,
        markerfacecolor="white",
        markeredgecolor=color,
        markeredgewidth=1.4,
        zorder=4
    )

    # --------------------------------------------------------
    # Predicted
    # --------------------------------------------------------

    ax.plot(
        tmp["x"],
        tmp["median_pred_c"],
        color=color,
        linestyle="--",
        linewidth=PRED_LINEWIDTH,
        marker="s",
        markersize=PRED_MARKERSIZE,
        markerfacecolor=color,
        markeredgecolor=color,
        markeredgewidth=1.0,
        zorder=3
    )


ax.set_title("")
ax.text(
    0.0, 1.02, "(a)",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontweight="bold",
    fontsize=11
)

ax.set_ylabel(
    "Median LST (°C)"
)

ax.set_xlabel(
    "Urban core and peripheral distance bands"
)

ax.set_xticks(
    x_a
)

ax.set_xticklabels(
    zone_labels_a
)

ax.set_xlim(
    -0.25,
    4.25
)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.7,
    alpha=0.30
)


# ------------------------------------------------------------
# Etiqueta R
# ------------------------------------------------------------

yl_a = ax.get_ylim()

ax.text(
    R_X_A,
    yl_a[1]
    - 0.025 * (
        yl_a[1] - yl_a[0]
    ),
    "R",
    ha="center",
    va="top",
    fontsize=10,
    fontweight="bold",
    color="0.25"
)


# ============================================================
# 11. PANEL (b)
# Peripheral thermal contrast
# ============================================================

ax = axes[1]

for year in [2023, 2024, 2025]:

    tmp = (
        plot_df_b[
            plot_df_b["year"] == year
        ]
        .sort_values("x")
    )

    color = year_colors[year]

    # --------------------------------------------------------
    # Observed
    # --------------------------------------------------------

    ax.plot(
        tmp["x"],
        tmp["contrast_obs_c"],
        color=color,
        linestyle="-",
        linewidth=OBS_LINEWIDTH,
        marker="o",
        markersize=OBS_MARKERSIZE,
        markerfacecolor="white",
        markeredgecolor=color,
        markeredgewidth=1.4,
        zorder=4
    )

    # --------------------------------------------------------
    # Predicted
    # --------------------------------------------------------

    ax.plot(
        tmp["x"],
        tmp["contrast_pred_c"],
        color=color,
        linestyle="--",
        linewidth=PRED_LINEWIDTH,
        marker="s",
        markersize=PRED_MARKERSIZE,
        markerfacecolor=color,
        markeredgecolor=color,
        markeredgewidth=1.0,
        zorder=3
    )


# ------------------------------------------------------------
# Línea cero = referencia térmica
# ------------------------------------------------------------

ax.axhline(
    0,
    color="0.30",
    linewidth=1.0,
    linestyle=":",
    zorder=1
)


ax.set_title("")
ax.text(
    0.0, 1.02, "(b)",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontweight="bold",
    fontsize=11
)

ax.set_ylabel(
    r"$\Delta$LST relative to $T_{ref}$ (°C)"
)

ax.set_xlabel(
    "Distance from consolidated built-up core"
)

ax.set_xticks(
    x_b
)

ax.set_xticklabels(
    zone_labels_b
)

ax.set_xlim(
    -0.25,
    3.25
)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.7,
    alpha=0.30
)


# ============================================================
# 12. AJUSTAR RANGO Y DEL PANEL (b)
# ============================================================

all_b = np.concatenate([
    plot_df_b["contrast_obs_c"].to_numpy(),
    plot_df_b["contrast_pred_c"].to_numpy()
])

ymin_b = float(
    np.nanmin(all_b)
)

ymax_b = float(
    np.nanmax(all_b)
)

yrange_b = (
    ymax_b - ymin_b
)

pad_b = (
    0.13 * yrange_b
)

ax.set_ylim(
    ymin_b - pad_b,
    ymax_b + pad_b
)


# ------------------------------------------------------------
# Etiqueta R
# ------------------------------------------------------------

yl_b = ax.get_ylim()

ax.text(
    R_X_B,
    yl_b[1]
    - 0.025 * (
        yl_b[1] - yl_b[0]
    ),
    "R",
    ha="center",
    va="top",
    fontsize=10,
    fontweight="bold",
    color="0.25"
)


# ============================================================
# 13. LEYENDA COMÚN SUPERIOR
# ============================================================

legend_handles = [

    # --------------------------------------------------------
    # Años
    # --------------------------------------------------------

    Line2D(
        [0], [0],
        color=year_colors[2023],
        linewidth=2.0,
        label="2023"
    ),

    Line2D(
        [0], [0],
        color=year_colors[2024],
        linewidth=2.0,
        label="2024"
    ),

    Line2D(
        [0], [0],
        color=year_colors[2025],
        linewidth=2.0,
        label="2025"
    ),

    # --------------------------------------------------------
    # Observed
    # --------------------------------------------------------

    Line2D(
        [0], [0],
        color="0.20",
        linestyle="-",
        linewidth=OBS_LINEWIDTH,
        marker="o",
        markersize=5,
        markerfacecolor="white",
        markeredgecolor="0.20",
        markeredgewidth=1.2,
        label="Observed"
    ),

    # --------------------------------------------------------
    # Predicted
    # --------------------------------------------------------

    Line2D(
        [0], [0],
        color="0.20",
        linestyle="--",
        linewidth=PRED_LINEWIDTH,
        marker="s",
        markersize=4.8,
        markerfacecolor="0.20",
        markeredgecolor="0.20",
        label="Predicted"
    ),
]


fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.985),
    ncol=5,
    frameon=False,
    handlelength=2.6,
    columnspacing=1.8
)


# ============================================================
# 14. GUARDAR FIGURA FINAL
# ============================================================

PNG_PATH = (
    FIG_DIR
    / "Urban_to_Peripheral_Thermal_Gradient_2023_2025_FINAL.png"
)

PDF_PATH = (
    FIG_DIR
    / "Urban_to_Peripheral_Thermal_Gradient_2023_2025_FINAL.pdf"
)


fig.savefig(
    PNG_PATH,
    dpi=EXPORT_DPI,
    bbox_inches="tight"
)

fig.savefig(
    PDF_PATH,
    dpi=EXPORT_DPI,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 15. RESUMEN
# ============================================================

print(
    "\nFIGURA FINAL GENERADA"
)

print(
    "================================"
)

print(
    "PNG:",
    PNG_PATH
)

print(
    "PDF:",
    PDF_PATH
)

print(
    "\nPanel (a):"
)

print(
    "Urban core → 0–2 → 2–4 → 4–6 → 6–8 km"
)

print(
    "\nPanel (b):"
)

print(
    "0–2 → 2–4 → 4–6 → 6–8 km"
)

print(
    "\nCodificación gráfica:"
)

print(
    "Color = TEST year"
)

print(
    "Solid circle = Observed"
)

print(
    "Dashed square = Predicted"
)

print(
    "\n8–10 km excluida deliberadamente."
)
